# Notebook 05 — Insights Estratégicos

Tres análisis accionables derivados del EDA:

1. **Impacto de descuentos por categoría** — puntos de quiebre y simulaciones
2. **Matriz BCG por sub-categoría** — priorización de portfolio
3. **Estacionalidad y optimización de márgenes** — calendario promocional


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv
import os
import pymysql

load_dotenv()
DB_USER = os.getenv('DB_USER')
DB_PASS = os.getenv('DB_PASS')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = int(os.getenv('DB_PORT', 3306))
DB_NAME = os.getenv('DB_NAME', 'sales_db')

def query(sql):
    conn = pymysql.connect(
        host=DB_HOST, port=DB_PORT,
        user=DB_USER, password=DB_PASS,
        database=DB_NAME, charset='utf8mb4')
    df = pd.read_sql(sql, conn)
    conn.close()
    return df

df = query('''
    SELECT fs.Sales, fs.Profit, fs.Discount, fs.Quantity,
           fs.Order_Date, fs.Region,
           dp.Category, dp.`Sub-Category` AS Sub_Category
    FROM fact_sales fs
    LEFT JOIN dim_product dp ON fs.Product_ID = dp.Product_ID
''')
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
df['year']  = df['Order_Date'].dt.year
df['month'] = df['Order_Date'].dt.month
print(f'Registros: {len(df):,}')

---
## Insight 1 — Impacto de Descuentos por Categoría

In [ ]:
RANGOS   = [0, 0.10, 0.20, 0.30, 0.40, 1.01]
ETIQUETAS = ['0-10%','10-20%','20-30%','30-40%','40%+']
df['disc_rango'] = pd.cut(df['Discount'], bins=RANGOS, labels=ETIQUETAS, right=False)

tabla = (df.groupby(['Category','disc_rango'], observed=True)
           .agg(n_ventas=('Sales','count'),
                profit_prom=('Profit','mean'),
                profit_total=('Profit','sum'),
                pct_perdidas=('Profit', lambda x: (x<0).mean()*100))
           .reset_index())

print('Profit promedio por categoria y rango de descuento:')
pivot = tabla.pivot(index='disc_rango', columns='Category', values='profit_prom').round(2)
print(pivot.to_string())

print('\nPunto de quiebre (primer rango con profit_prom < 0):')
for cat in df['Category'].unique():
    sub = tabla[tabla['Category']==cat].copy()
    negativo = sub[sub['profit_prom'] < 0]
    punto = negativo.iloc[0]['disc_rango'] if len(negativo) else 'Sin quiebre'
    print(f'  {cat:<22}: {punto}')

In [ ]:
cats   = df['Category'].unique()
colors = {'Technology':'#4C78A8','Furniture':'#F58518','Office Supplies':'#72B7B2'}

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)
fig.suptitle('Profit promedio por rango de descuento y categoria', fontsize=13, fontweight='bold')

for ax, cat in zip(axes, cats):
    sub = tabla[tabla['Category']==cat]
    barras = ax.bar(sub['disc_rango'], sub['profit_prom'],
                    color=[('#E45756' if v < 0 else colors[cat]) for v in sub['profit_prom']],
                    alpha=0.85, edgecolor='white')
    ax.axhline(0, color='black', linewidth=1)
    for bar, row in zip(barras, sub.itertuples()):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                f'n={row.n_ventas}', ha='center', fontsize=7)
    ax.set_title(cat, fontweight='bold')
    ax.set_xlabel('Rango descuento')
    ax.set_ylabel('Profit promedio ($)')
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../outputs/ins_01_profit_descuento.png', bbox_inches='tight')
plt.show()

In [ ]:
# Escenarios: limitar descuento maximo por categoria
ESCENARIOS = [
    ('Furniture',       0.10, 'A'),
    ('Technology',      0.20, 'B'),
    ('Office Supplies', 0.15, 'C'),
]

print('SIMULACION DE ESCENARIOS\n' + '='*60)
for cat, limite, letra in ESCENARIOS:
    sub = df[df['Category']==cat]
    afectadas = sub[sub['Discount'] > limite]
    profit_actual  = sub['Profit'].sum()
    profit_perdido = afectadas['Profit'].sum()           # lo que generaron (negativo o bajo)
    revenue_en_riesgo = afectadas['Sales'].sum()
    n_afect = len(afectadas)

    # Ganancia si esas ventas se hubieran hecho al limite
    # Aproximacion: profit = sales * margen_promedio_al_limite
    sub_limite = sub[sub['Discount'] <= limite]
    margen_lim = sub_limite['Profit'].sum() / sub_limite['Sales'].sum() if sub_limite['Sales'].sum() > 0 else 0
    profit_simulado = afectadas['Sales'].sum() * margen_lim
    mejora = profit_simulado - profit_perdido

    print(f'Escenario {letra} — {cat} (max {limite*100:.0f}%)')
    print(f'  Ventas afectadas          : {n_afect:,} ({n_afect/len(sub)*100:.1f}%)')
    print(f'  Revenue en riesgo         : ${revenue_en_riesgo:,.0f}')
    print(f'  Profit actual esas ventas : ${profit_perdido:,.0f}')
    print(f'  Profit simulado al limite : ${profit_simulado:,.0f}')
    print(f'  Mejora de profit          : ${mejora:,.0f}')
    print()

In [ ]:
# ROI del descuento: por cada $1 de descuento otorgado, cuanto sales adicional se genera?
print('ROI DE DESCUENTOS\n' + '='*60)
for cat in df['Category'].unique():
    sub = df[(df['Category']==cat) & (df['Discount'] > 0)].copy()
    sub['monto_descuento'] = sub['Sales'] / (1 - sub['Discount']) * sub['Discount']
    roi = sub['Sales'].sum() / sub['monto_descuento'].sum()
    margin = sub['Profit'].sum() / sub['Sales'].sum() * 100
    print(f'  {cat:<22}: ${roi:.2f} sales por $1 de descuento | margen: {margin:.1f}%')

print()
print('LIMITES RECOMENDADOS POR CATEGORIA:')
print('  Furniture       : max 10%  (quiebre en 20-30%, descuentos >30% destruyen valor)')
print('  Technology      : max 20%  (quiebre en 30-40%, margen alto permite hasta 20%)')
print('  Office Supplies : max 15%  (quiebre en 20-30%, categoria sensible al descuento)')

---
## Insight 2 — Matriz BCG por Sub-Categoría

In [ ]:
# CAGR 2020-2023 y metricas por sub-categoria
yrly = df.groupby(['year','Sub_Category'])['Sales'].sum().reset_index()

def cagr(sub):
    pivoted = sub.set_index('year')['Sales']
    years_present = sorted(pivoted.index)
    if len(years_present) < 2: return 0
    v0, v1 = pivoted[years_present[0]], pivoted[years_present[-1]]
    n = years_present[-1] - years_present[0]
    return ((v1/v0)**(1/n) - 1)*100 if v0 > 0 else 0

cagr_s = yrly.groupby('Sub_Category').apply(cagr).reset_index()
cagr_s.columns = ['Sub_Category','CAGR']

total = df.groupby('Sub_Category').agg(
    sales_total=('Sales','sum'),
    profit_total=('Profit','sum'),
    margen=('Profit', lambda x: x.sum()/df.loc[x.index,'Sales'].sum()*100),
).reset_index()
total['pct_profit'] = total['profit_total'] / total['profit_total'].sum() * 100

bcg = total.merge(cagr_s, on='Sub_Category')

# Clasificacion BCG
med_cagr   = bcg['CAGR'].median()
med_profit = bcg['pct_profit'].median()

def clasif(row):
    hi_g = row['CAGR']       >= med_cagr
    hi_p = row['pct_profit'] >= med_profit
    if hi_g and hi_p:  return 'Star'
    if not hi_g and hi_p: return 'Cash Cow'
    if hi_g and not hi_p: return 'Question Mark'
    return 'Dog'

bcg['BCG'] = bcg.apply(clasif, axis=1)

print('TABLA BCG POR SUB-CATEGORIA')
print(bcg[['Sub_Category','BCG','CAGR','pct_profit','profit_total','margen']]
      .sort_values('BCG').round(2).to_string(index=False))

In [ ]:
# Grafico BCG Matrix
fig, ax = plt.subplots(figsize=(12, 8))
color_map = {'Star':'#F58518','Cash Cow':'#4C78A8','Question Mark':'#72B7B2','Dog':'#E45756'}
emoji_map  = {'Star':'★','Cash Cow':'$','Question Mark':'?','Dog':'✗'}

for _, row in bcg.iterrows():
    c = color_map[row['BCG']]
    sz = max(row['sales_total']/5000, 50)
    ax.scatter(row['CAGR'], row['pct_profit'], s=sz, color=c, alpha=0.75, edgecolors='white', linewidth=1.5)
    ax.annotate(f"{emoji_map[row['BCG']]} {row['Sub_Category']}",
                (row['CAGR'], row['pct_profit']),
                fontsize=7.5, ha='center', va='bottom')

ax.axvline(med_cagr,   color='gray', linestyle='--', linewidth=1)
ax.axhline(med_profit, color='gray', linestyle='--', linewidth=1)
ax.set_xlabel('CAGR 2020-2023 (%)', fontsize=11)
ax.set_ylabel('% del Profit Total', fontsize=11)
ax.set_title('Matriz BCG — Sub-Categorias', fontsize=13, fontweight='bold')

patches = [mpatches.Patch(color=v, label=k) for k,v in color_map.items()]
ax.legend(handles=patches, loc='upper left', fontsize=9)

for (x, y, lbl) in [(med_cagr+1, bcg['pct_profit'].max()*0.95,'STAR / QUESTION'),
                     (bcg['CAGR'].min()*0.9, bcg['pct_profit'].max()*0.95,'CASH COW / DOG')]:
    ax.text(x, y, lbl, fontsize=8, color='gray')

plt.tight_layout()
plt.savefig('../outputs/ins_02_bcg_matrix.png', bbox_inches='tight')
plt.show()

In [ ]:
# Recomendaciones BCG y productos problematicos
print('RECOMENDACIONES POR CUADRANTE\n' + '='*60)
for cuadrante in ['Star','Cash Cow','Question Mark','Dog']:
    productos = bcg[bcg['BCG']==cuadrante]['Sub_Category'].tolist()
    print(f'\n{cuadrante}: {productos}')

print('\nPRODUCTOS CON PROFIT NEGATIVO TOTAL:')
negativos = bcg[bcg['profit_total'] < 0].sort_values('profit_total')
for _, r in negativos.iterrows():
    print(f'  {r["Sub_Category"]:<22}: Profit total = ${r["profit_total"]:,.0f} | CAGR={r["CAGR"]:.1f}%')

print('\nANALISIS Tables y Bookcases (por que pierden?):')
for sc in ['Tables','Bookcases']:
    sub = df[df['Sub_Category']==sc]
    if len(sub) == 0: continue
    disc_prom = sub['Discount'].mean()*100
    margin    = sub['Profit'].sum()/sub['Sales'].sum()*100
    pct_neg   = (sub['Profit']<0).mean()*100
    print(f'  {sc}: descuento prom={disc_prom:.1f}% | margen={margin:.1f}% | {pct_neg:.0f}% ventas con perdida')

---
## Insight 3 — Estacionalidad y Optimización de Márgenes

In [ ]:
# Metricas mensuales
mes_nombres = {1:'Ene',2:'Feb',3:'Mar',4:'Abr',5:'May',6:'Jun',
               7:'Jul',8:'Ago',9:'Sep',10:'Oct',11:'Nov',12:'Dic'}

monthly = df.groupby('month').agg(
    sales=('Sales','sum'),
    profit=('Profit','sum'),
    disc_prom=('Discount','mean'),
    n=('Sales','count')
).reset_index()
monthly['margen'] = monthly['profit'] / monthly['sales'] * 100
monthly['mes']    = monthly['month'].map(mes_nombres)

print('TABLA MENSUAL: ventas, profit, margen, descuento promedio')
print(monthly[['mes','sales','profit','margen','disc_prom','n']]
      .rename(columns={'disc_prom':'desc_prom'})
      .round(2).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9))
fig.suptitle('Estacionalidad: Profit y Descuentos por Mes', fontsize=13, fontweight='bold')

# Grafico 1: Sales y Profit por mes
ax = axes[0]
x = monthly['mes']
ax.bar(x, monthly['sales'],  alpha=0.5, color='#4C78A8', label='Sales')
ax.bar(x, monthly['profit'], alpha=0.85, color='#F58518', label='Profit')
ax.axhline(0, color='black', linewidth=0.8)
ax2 = ax.twinx()
ax2.plot(x, monthly['margen'], 'o-', color='#E45756', linewidth=2, label='Margen %')
ax2.set_ylabel('Margen %', color='#E45756')
ax.set_ylabel('$ (Sales / Profit)')
ax.set_title('Sales, Profit y Margen mensual', fontweight='bold')
ax.legend(loc='upper left')
ax2.legend(loc='upper right')

# Grafico 2: Descuento promedio por mes
ax = axes[1]
colores = ['#E45756' if d > 0.20 else '#4C78A8' for d in monthly['disc_prom']]
ax.bar(x, monthly['disc_prom']*100, color=colores, alpha=0.85, edgecolor='white')
ax.axhline(20, color='red', linestyle='--', linewidth=1.2, label='Limite 20%')
ax.set_ylabel('Descuento promedio (%)')
ax.set_title('Descuento promedio mensual (rojo = supera 20%)', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig('../outputs/ins_03_estacionalidad.png', bbox_inches='tight')
plt.show()

In [ ]:
# Misterio de Marzo: que lo hace especial
print('ANALISIS MARZO vs RESTO DEL ANO\n' + '='*55)
marzo = df[df['month']==3]
resto = df[df['month']!=3]
print(f'Ventas promedio: Marzo=${marzo["Sales"].mean():.0f} | Resto=${resto["Sales"].mean():.0f}')
print(f'Margen:          Marzo={marzo["Profit"].sum()/marzo["Sales"].sum()*100:.1f}% | Resto={resto["Profit"].sum()/resto["Sales"].sum()*100:.1f}%')
print(f'Descuento prom: Marzo={marzo["Discount"].mean()*100:.1f}% | Resto={resto["Discount"].mean()*100:.1f}%')
print(f'\nTop categorias en Marzo:')
print(marzo.groupby('Category')['Sales'].sum().sort_values(ascending=False).to_string())
print(f'\nTop sub-categorias en Marzo:')
print(marzo.groupby('Sub_Category')['Sales'].sum().sort_values(ascending=False).head(5).to_string())

# Paradoja Q4
print('\nPARADOJA Q4 (Nov/Dic alto volumen, bajo margen)\n' + '='*55)
for m, nombre in [(11,'Noviembre'),(12,'Diciembre')]:
    sub = df[df['month']==m]
    disc_actual = sub['Discount'].mean()
    margen_actual = sub['Profit'].sum()/sub['Sales'].sum()*100
    # Simular descuento limitado a 15%
    disc_lim = 0.15
    afectadas = sub[sub['Discount'] > disc_lim]
    resto_mes = sub[sub['Discount'] <= disc_lim]
    margen_resto = (resto_mes['Profit'].sum()/resto_mes['Sales'].sum()*100) if len(resto_mes) else 0
    profit_simulado = afectadas['Sales'].sum() * margen_resto/100
    mejora = profit_simulado - afectadas['Profit'].sum()
    print(f'{nombre}: desc_prom={disc_actual*100:.1f}% | margen actual={margen_actual:.1f}%')
    print(f'  Si limitamos a 15%: {len(afectadas)} ventas afectadas, mejora profit=${mejora:,.0f}')

In [ ]:
# Calendario de descuentos recomendados
print('CALENDARIO PROMOCIONAL RECOMENDADO\n' + '='*60)
print(f'{"Mes":<12} {"Desc actual%":>13} {"Margen actual%":>15} {"Desc max rec%":>14} {"Accion"}')
print('-'*70)

def recomendacion(margen, disc):
    if margen >= 20 and disc <= 0.15: return 'Mantener',     15
    if margen >= 15 and disc <= 0.20: return 'Mantener',     20
    if margen < 10:                   return 'Reducir desc', 10
    if disc > 0.20:                   return 'Reducir desc', 15
    return 'Evaluar', int(disc*100)

for _, row in monthly.iterrows():
    accion, max_desc = recomendacion(row['margen'], row['disc_prom'])
    alerta = ' ⚠' if row['disc_prom']*100 > max_desc else ''
    print(f'{row["mes"]:<12} {row["disc_prom"]*100:>12.1f}% {row["margen"]:>14.1f}% {max_desc:>13}%   {accion}{alerta}')

---
## Resumen ejecutivo

### Hallazgo 1 — Descuentos
- **Furniture**: quiebre de rentabilidad entre 20-30%. Límite recomendado: **10%**
- **Technology**: soporta hasta 20% de descuento con margen positivo
- **Office Supplies**: sensible al descuento, límite recomendado: **15%**

### Hallazgo 2 — Portfolio (BCG)
- **Stars**: sub-categorías con alto crecimiento y profit — priorizar inversión
- **Dogs con profit negativo**: Tables y Bookcases — revisar estructura de precios o descontinuar
- Aplicar descuentos solo en Stars y Cash Cows; eliminar descuentos en Dogs

### Hallazgo 3 — Estacionalidad
- **Marzo**: margen superior al promedio — replicar condiciones (menos descuentos, categorías de alto valor)
- **Q4 (Nov/Dic)**: paradoja de alto volumen + bajo margen por descuentos excesivos. Limitar a 15% mejora profit significativamente
- **Enero**: revisar descuentos de inicio de año que erosionan margen
